# Module 11 — Transformer Forward Pass as GPU Workloads

Inspect Qwen2.5-0.5B projection shapes, calculate FLOPs and weight bytes, capture real intermediate tensors, benchmark 1-token versus 128-token projections, and profile one forward pass on a T4.

Install dependencies if needed:

```bash
pip install -U 'transformers>=4.45,<5' accelerate safetensors pandas
```

In [ ]:
# Complete standalone Transformer workload lab; no import from the companion .py file.
import statistics
import pandas as pd
import torch
from transformers import AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
assert torch.cuda.is_available(), 'Select a T4 GPU runtime.'
device = torch.device('cuda:0')
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device).eval()

def projection_modules(model):
    layer = model.model.layers[0]
    return {'q_proj': layer.self_attn.q_proj, 'k_proj': layer.self_attn.k_proj, 'v_proj': layer.self_attn.v_proj, 'o_proj': layer.self_attn.o_proj, 'gate_proj': layer.mlp.gate_proj, 'up_proj': layer.mlp.up_proj, 'down_proj': layer.mlp.down_proj}

def linear_stats(module, tokens):
    out_features, in_features = module.weight.shape; flops = 2 * tokens * in_features * out_features; weight_bytes = module.weight.numel() * module.weight.element_size()
    return {'gflops': flops / 1e9, 'weight_mib': weight_bytes / 2**20, 'flops_per_weight_byte': flops / weight_bytes}

def benchmark_module(module, x, device, warmup=5, repeats=30):
    with torch.inference_mode():
        for _ in range(warmup): module(x)
        torch.cuda.synchronize(device); samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True); end = torch.cuda.Event(enable_timing=True); start.record(); module(x); end.record(); end.synchronize(); samples.append(start.elapsed_time(end))
    return statistics.median(samples)

def inspect_hooks(model, device, tokens=128):
    modules = projection_modules(model); captured = []; handles = []
    def make_hook(name):
        def hook(module, inputs, output): captured.append({'module': name, 'input_shape': tuple(inputs[0].shape), 'output_shape': tuple(output.shape), 'dtype': str(output.dtype)})
        return hook
    for name, module in modules.items(): handles.append(module.register_forward_hook(make_hook(name)))
    input_ids = torch.randint(100, 50000, (1, tokens), device=device)
    with torch.inference_mode(): model(input_ids=input_ids, use_cache=False, return_dict=True)
    for handle in handles: handle.remove()
    return captured

print('GPU:', torch.cuda.get_device_name(device))
print('Hidden size:', model.config.hidden_size)
print('Layers:', model.config.num_hidden_layers)
print('Query/KV heads:', model.config.num_attention_heads, model.config.num_key_value_heads)
print('MLP intermediate:', model.config.intermediate_size)

In [ ]:
# Inspect the seven large linear projections in the first Transformer layer.
# FLOPs use 2 * tokens * input_features * output_features for a multiply-add pair.
modules = projection_modules(model)
rows = []
for name, module in modules.items():
    stats = linear_stats(module, tokens=128)
    rows.append({
        'module': name,
        'weight_shape': tuple(module.weight.shape),
        'GFLOPs': stats['gflops'],
        'weight_MiB': stats['weight_mib'],
        'FLOPs_per_weight_byte': stats['flops_per_weight_byte'],
    })
display(pd.DataFrame(rows))

In [ ]:
# Forward hooks observe real tensors without modifying the model's forward implementation.
# The synthetic IDs are only for shape inspection; their token meanings are irrelevant.
captured = inspect_hooks(model, device, tokens=128)
display(pd.DataFrame(captured))

In [ ]:
# Compare a decode-like one-token shape with a prefill-like 128-token shape.
# CUDA events measure device execution rather than only asynchronous Python dispatch.
timing_rows = []
for tokens in [1, 128]:
    for name, module in modules.items():
        input_features = module.weight.shape[1]
        x = torch.randn(1, tokens, input_features, device=device, dtype=torch.float16)
        milliseconds = benchmark_module(module, x, device, warmup=5, repeats=30)
        flops = 2 * tokens * input_features * module.weight.shape[0]
        timing_rows.append({
            'module': name, 'tokens': tokens, 'ms': milliseconds,
            'TFLOPS': flops / (milliseconds / 1000) / 1e12,
        })
timing_df = pd.DataFrame(timing_rows)
display(timing_df)

In [ ]:
# Profile one warmed-up forward pass to map conceptual stages to PyTorch/CUDA operators.
# Profiling adds overhead, so use this for attribution rather than final benchmarking.
from torch.profiler import ProfilerActivity, profile
input_ids = torch.randint(100, 50000, (1, 128), device=device)
with torch.inference_mode():
    for _ in range(3):
        model(input_ids=input_ids, use_cache=False)
torch.cuda.synchronize()
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True) as prof:
    with torch.inference_mode():
        model(input_ids=input_ids, use_cache=False, return_dict=True)
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by='self_cuda_time_total', row_limit=20))

## Interpretation

Compare 1-token and 128-token projection efficiency. Larger token dimensions can improve weight reuse and GEMM utilization, while attention and KV-cache behavior remain context-dependent. The profiler output maps conceptual Transformer stages to actual PyTorch operators and CUDA kernels.